# Backtester Core Engine

**Owner:** James  
**Version:** 2.0 (Feb 13, 2026)  

Core backtesting engine for evaluating sports betting strategies.  
All logic is imported from `src/cuic_quant/backtest/backtester_backend.py`.  
See `docs/reference/strategy-interface.md` for the strategy function contract.

## Output Format

| Column | Type | Description |
|--------|------|-------------|
| timestamp | datetime | When trade happened |
| game | str | "Home vs Away" |
| action | str | 'BUY_HOME' or 'BUY_AWAY' |
| bet_size | float | Dollars bet |
| odds | float | Decimal odds used |
| outcome | str | 'WIN' or 'LOSS' |
| pnl | float | Profit/loss for this trade |
| cumulative_pnl | float | Running total P&L |
| bankroll | float | Current bankroll after trade |

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is importable when running from tools/ directory
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "tools":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from cuic_quant.backtest import (
    load_backtest_data,
    backtest,
    always_bet_home,
    validate_backtest_results,
)

## Configuration

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
DUMMY_CSV = DATA_DIR / "dummy_backtest_input.csv"
TEST_CSV = DATA_DIR / "test_games.csv"

## Load Data

In [ ]:
data = load_backtest_data("2026-01-01", "2026-01-31", csv_path=DUMMY_CSV)
print(f"Games loaded: {len(data)}")
data.head()

## Strategy

Using the built-in `always_bet_home` test strategy.

To use your own strategy, replace the assignment below:

```python
def my_strategy(row, context=None):
    return {"action": "BUY_HOME", "confidence": 0.8, "size": 50.0}

strategy = my_strategy
```

See `docs/reference/strategy-interface.md` for the full contract.

In [ ]:
strategy = always_bet_home

## Run Backtest

In [ ]:
results = backtest(data, strategy, initial_bankroll=10000.0)
print(f"Total trades: {len(results)}")
results

## Summary Statistics

In [ ]:
if len(results) > 0:
    wins = (results["outcome"] == "WIN").sum()
    losses = (results["outcome"] == "LOSS").sum()
    win_rate = wins / len(results)
    final_pnl = results["cumulative_pnl"].iloc[-1]
    final_bankroll = results["bankroll"].iloc[-1]

    print(f"Win Rate:        {win_rate:.1%} ({wins}W / {losses}L)")
    print(f"Total P&L:       ${final_pnl:,.2f}")
    print(f"Final Bankroll:  ${final_bankroll:,.2f}")
    print(f"ROI:             {final_pnl / 10000:.1%}")
else:
    print("No trades executed.")

## Validate Results

Run the full validation suite to check schema, math correctness, and data leakage.

In [ ]:
report = validate_backtest_results(results, data)

if report["passed"]:
    print(f"PASSED: {report['checks_passed']}/{report['checks_run']} checks passed")
else:
    print(f"FAILED: {report['checks_passed']}/{report['checks_run']} checks passed")
    for f in report["failures"]:
        print(f"  - {f}")

## Test with Mya's `test_games.csv`

Run the backtester against Mya's 100-row test dataset to validate compatibility.

In [ ]:
if TEST_CSV.exists():
    mya_data = load_backtest_data("2026-01-01", "2026-12-31", csv_path=TEST_CSV)
    mya_results = backtest(mya_data, strategy, initial_bankroll=10000.0)
    mya_report = validate_backtest_results(mya_results, mya_data)

    print(f"Mya's data: {len(mya_results)} trades")
    print(f"Validation: {'PASSED' if mya_report['passed'] else 'FAILED'}")
    print(f"Checks: {mya_report['checks_passed']}/{mya_report['checks_run']}")
else:
    print(f"test_games.csv not found at {TEST_CSV}")